In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score, 
    roc_auc_score, 
    roc_curve, 
    auc
)
from sklearn.preprocessing import label_binarize

# 1. Load Data and Models
data = np.load('fashion_data_complete.npz')
X_test = data['X_test']
y_test = data['y_test']
class_names = data['class_names']

# Dictionary of your 4 models (Update filenames if needed)
model_files = {
    "k-NN": "knn_fashion_model.joblib",
    "Naive Bayes": "nb_fashion_model.joblib",
    "SVM": "svm_fashion_model.joblib",
    "Logistic Regression": "logistic_model.joblib" # Assuming Hieu's model name
}

models = {}
for name, file in model_files.items():
    try:
        models[name] = joblib.load(file)
        print(f"Loaded {name}")
    except:
        print(f"Warning: Could not find {file}")



In [ ]:
# 2. Binarize labels for Multi-class ROC-AUC
y_test_binarized = label_binarize(y_test, classes=range(10))

# 3. Evaluation Loop
results_summary = []

plt.figure(figsize=(15, 12)) # For Confusion Matrices

for i, (name, model) in enumerate(models.items()):
    # Get Predictions
    y_pred = model.predict(X_test)
    
    # Get Probabilities (for ROC-AUC)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)
    else:
        # For SVM if probability=False, use decision_function
        y_score = model.decision_function(X_test)
        # Normalize scores to look like probabilities for the metric
        y_score = (y_score - y_score.min()) / (y_score.max() - y_score.min())

    # --- Metrics ---
    acc = accuracy_score(y_test, y_pred)
    # Multi-class ROC-AUC (Macro average)
    roc_auc = roc_auc_score(y_test, y_score, multi_class='ovr', average='macro')
    
    results_summary.append({
        "Model": name,
        "Accuracy": acc,
        "ROC-AUC (Macro)": roc_auc
    })

    # --- Print Classification Report ---
    print(f"\n{'='*20} {name} {'='*20}")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # --- Plot Confusion Matrix ---
    plt.subplot(2, 2, i+1)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix: {name}")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')

plt.tight_layout()
plt.show()


In [ ]:

# 4. Show Summary Table
df_compare = pd.DataFrame(results_summary).sort_values(by='Accuracy', ascending=False)
print("\n--- Model Comparison Summary ---")
print(df_compare)



In [ ]:
# 5. Plot ROC Curves (Macro-Average for each model)
plt.figure(figsize=(10, 7))
for name, model in models.items():
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)
    else:
        y_score = model.decision_function(X_test)
    
    # Compute ROC curve and ROC area for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for j in range(10):
        fpr[j], tpr[j], _ = roc_curve(y_test_binarized[:, j], y_score[:, j])
        roc_auc[j] = auc(fpr[j], tpr[j])

    # Compute macro-average ROC curve
    all_fpr = np.unique(np.concatenate([fpr[j] for j in range(10)]))
    mean_tpr = np.zeros_like(all_fpr)
    for j in range(10):
        mean_tpr += np.interp(all_fpr, fpr[j], tpr[j])
    mean_tpr /= 10
    
    plt.plot(all_fpr, mean_tpr, label=f'{name} (area = {auc(all_fpr, mean_tpr):.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC Curves (Macro-Average)')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()